# CPU Precision Benchmark — Local Laptop
Reuses the exact same benchmark methodology as the GPU-machine notebook (10 warm-up passes, 100 timed passes, batch size 1, same 70/15/15 seeded split), run here on **this laptop's CPU only**, for all three precisions.

**Assumption:** "FP32 / FP16 / INT8" — the same three variants used throughout the rest of the study. If you actually meant a different precision set, let me know and I'll adjust this notebook.

**Before running:** copy the `checkpoints/` folder (containing `variant_fp32.pt`, `variant_fp16.pt`, `variant_int8.pt`) from your GPU machine onto this laptop, and update `CHECKPOINT_DIR` below to point at it. The EuroSAT dataset will download automatically on first run if not already present.

**Why this run matters:** the CPU numbers in your current results.csv were measured on the same desktop as your RTX 5060 GPU — not a laptop. This notebook gives you an independent CPU data point on genuinely different hardware, which is worth having before you generalize any CPU-related claim (especially the CPU/FP16 slowdown finding).

In [1]:
import sys, platform, torch, torchvision, sklearn, pandas, psutil
print(f"Python:      {sys.version}")
print(f"Platform:    {platform.platform()}")
print(f"Machine:     {platform.machine()}")
print(f"PyTorch:     {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"Scikit-learn:{sklearn.__version__}")
print(f"Pandas:      {pandas.__version__}")
print(f"CPU cores:   {psutil.cpu_count(logical=False)} physical / {psutil.cpu_count(logical=True)} logical")
print(f"RAM:         {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"\nCUDA available: {torch.cuda.is_available()}  (expected False — this notebook is CPU-only by design)")

Python:      3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]
Platform:    Windows-10-10.0.26200-SP0
Machine:     AMD64
PyTorch:     2.11.0+cpu
Torchvision: 0.26.0+cpu
Scikit-learn:1.8.0
Pandas:      3.0.1
CPU cores:   4 physical / 8 logical
RAM:         15.7 GB

CUDA available: False  (expected False — this notebook is CPU-only by design)


## Fill in your exact CPU model here
Check Task Manager (Windows) / `lscpu` (Linux) / About This Mac (macOS) and record it — this goes in Section 5.6 of the paper alongside the GPU machine's spec.

`CPU_MODEL = "[fill in]"`

In [2]:
CPU_MODEL = "[fill in: your exact CPU model]"  # e.g. "Intel Core i5-1135G7"
print(f"Recording CPU model: {CPU_MODEL}")

Recording CPU model: [fill in: your exact CPU model]


## Cell 2 — Quantization backend (architecture-dependent)

In [3]:
import torch
import platform

print("PyTorch:", torch.__version__)
print("Machine:", platform.machine())
print("Supported quantized engines:", torch.backends.quantized.supported_engines)
print("Current engine:", torch.backends.quantized.engine)

PyTorch: 2.11.0+cpu
Machine: AMD64
Supported quantized engines: ['onednn']
Current engine: onednn


In [4]:
import torch
import platform

machine = platform.machine().lower()
supported = torch.backends.quantized.supported_engines

print("PyTorch:", torch.__version__)
print("Machine:", machine)
print("Supported quantized engines:", supported)

# Select an actually available quantization backend
if "onednn" in supported:
    torch.backends.quantized.engine = "onednn"
    print("Using oneDNN quantized backend.")
elif "fbgemm" in supported:
    torch.backends.quantized.engine = "fbgemm"
    print("Using FBGEMM quantized backend.")
elif "qnnpack" in supported:
    torch.backends.quantized.engine = "qnnpack"
    print("Using QNNPACK quantized backend.")
else:
    raise RuntimeError(
        f"No supported quantized backend found. Available: {supported}"
    )

PyTorch: 2.11.0+cpu
Machine: amd64
Supported quantized engines: ['onednn']
Using oneDNN quantized backend.


## Cell 3 — Paths and config

In [5]:
from pathlib import Path

CHECKPOINT_DIR = Path("./checkpoints")  # <-- point this at your copied checkpoints folder
DATA_ROOT = Path("./data")              # EuroSAT downloads here automatically if not present
RESULTS_CSV = Path("./results_cpu_laptop.csv")

SEED = 42
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
WARMUP_PASSES = 10
TIMED_PASSES = 100

variants = {
    "fp32": CHECKPOINT_DIR / "variant_fp32.pt",
    "fp16": CHECKPOINT_DIR / "variant_fp16.pt",
    "int8": CHECKPOINT_DIR / "variant_int8.pt",
}
for name, path in variants.items():
    status = "found" if path.exists() else "MISSING -- copy it from the GPU machine"
    print(f"{name}: {path}  ({status})")

fp32: checkpoints\variant_fp32.pt  (MISSING -- copy it from the GPU machine)
fp16: checkpoints\variant_fp16.pt  (MISSING -- copy it from the GPU machine)
int8: checkpoints\variant_int8.pt  (MISSING -- copy it from the GPU machine)


## Cell 4 — Load the test split (identical seed/split to every other run)

In [6]:
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
full_dataset = datasets.EuroSAT(root=str(DATA_ROOT), download=True, transform=transform)
n = len(full_dataset)
n_train = int(0.70 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val
generator = torch.Generator().manual_seed(SEED)  # same seed as every other run -> same test images
_, _, test_set = random_split(full_dataset, [n_train, n_val, n_test], generator=generator)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False)
print(f"Test set: {len(test_set)} images (same split used across the whole study)")

Test set: 4050 images (same split used across the whole study)


## Cell 5 — Load a variant

In [7]:
def load_variant(precision, device=torch.device("cpu")):
    ckpt = torch.load(variants[precision], map_location=device)
    if precision == "int8":
        model = ckpt["model"]  # already a full quantized nn.Module
    else:
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, len(ckpt["classes"]))
        model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model.to(device) if precision != "int8" else model, ckpt["classes"]

## Cell 6 — Benchmark harness (identical methodology to the GPU-machine notebook)

In [8]:
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def run_benchmark(precision):
    device = torch.device("cpu")
    model, classes = load_variant(precision, device)

    def to_input_dtype(x):
        return x.half() if precision == "fp16" else x

    it = iter(test_loader)
    with torch.no_grad():
        for _ in range(WARMUP_PASSES):
            images, _ = next(it)
            model(to_input_dtype(images).to(device))

    latencies = []
    with torch.no_grad():
        for _ in range(TIMED_PASSES):
            images, _ = next(it)
            images = to_input_dtype(images).to(device)
            t0 = time.perf_counter()
            model(images)
            latencies.append(time.perf_counter() - t0)

    mean_latency_ms = (sum(latencies) / len(latencies)) * 1000
    throughput_ips = 1000 / mean_latency_ms

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = to_input_dtype(images).to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(1).cpu().tolist())
            all_labels.extend(labels.tolist())

    memory_mb = psutil.Process().memory_info().rss / (1024 ** 2)

    return {
        "platform": f"CPU (laptop: {CPU_MODEL})",
        "precision": precision,
        "accuracy": round(accuracy_score(all_labels, all_preds), 4),
        "precision_m": round(precision_score(all_labels, all_preds, average="macro", zero_division=0), 4),
        "recall": round(recall_score(all_labels, all_preds, average="macro", zero_division=0), 4),
        "f1": round(f1_score(all_labels, all_preds, average="macro", zero_division=0), 4),
        "latency_ms": round(mean_latency_ms, 3),
        "throughput_ips": round(throughput_ips, 2),
        "memory_mb": round(memory_mb, 1),
    }

print("Benchmark harness defined.")
print(f"Warm-up: {WARMUP_PASSES} passes (discarded)   Timed: {TIMED_PASSES} passes")

Benchmark harness defined.
Warm-up: 10 passes (discarded)   Timed: 100 passes


## Cell 7 — Run all three precisions
This will take a while on FP16 in particular — based on the GPU-machine run, CPU/FP16 was ~51x slower than CPU/FP32, so expect that pattern to (possibly) repeat here. That's exactly the thing this notebook exists to check.

In [9]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nCheckpoints folder:")
checkpoints = Path("checkpoints")

if checkpoints.exists():
    for f in checkpoints.iterdir():
        print(f)
else:
    print("❌ checkpoints folder does not exist")

Current working directory:
C:\Users\BILL\Downloads

Checkpoints folder:
checkpoints\baseline_fp32.pt
checkpoints\train_resume.pt
checkpoints\variants


In [10]:
from pathlib import Path

variants_dir = Path("checkpoints") / "variants"

print("Files inside variants:")
for f in variants_dir.iterdir():
    print(f)

Files inside variants:
checkpoints\variants\variant_fp16.pt
checkpoints\variants\variant_fp32.pt
checkpoints\variants\variant_int8.pt
checkpoints\variants\variant_int8_ts.pt


In [11]:
from pathlib import Path

VARIANTS_DIR = Path("checkpoints") / "variants"

variants = {
    "fp32": VARIANTS_DIR / "variant_fp32.pt",
    "fp16": VARIANTS_DIR / "variant_fp16.pt",
    "int8": VARIANTS_DIR / "variant_int8.pt",
}

print("Checking model files...")

for precision, path in variants.items():
    print(f"{precision.upper():5} -> {path} -> exists={path.exists()}")

Checking model files...
FP32  -> checkpoints\variants\variant_fp32.pt -> exists=True
FP16  -> checkpoints\variants\variant_fp16.pt -> exists=True
INT8  -> checkpoints\variants\variant_int8.pt -> exists=True


In [12]:
import torch
import time
from pathlib import Path

print("1. PyTorch loaded")

path = Path("checkpoints/variants/variant_fp32.pt")
print("2. Checkpoint exists:", path.exists())

print("3. Loading checkpoint...")

start = time.time()

ckpt = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print(f"4. Checkpoint loaded in {time.time() - start:.2f} seconds")

print("5. Checkpoint type:", type(ckpt))

if isinstance(ckpt, dict):
    print("6. Checkpoint keys:", ckpt.keys())

1. PyTorch loaded
2. Checkpoint exists: True
3. Loading checkpoint...
4. Checkpoint loaded in 0.05 seconds
5. Checkpoint type: <class 'dict'>
6. Checkpoint keys: dict_keys(['model_state', 'classes', 'hyperparams'])


In [13]:
print("Loading through load_variant()...")

start = time.time()

model, classes = load_variant(
    "fp32",
    torch.device("cpu")
)

print(f"Model loaded in {time.time() - start:.2f} seconds")
print("Model type:", type(model))
print("Classes:", len(classes))

Loading through load_variant()...
Model loaded in 0.35 seconds
Model type: <class 'torchvision.models.resnet.ResNet'>
Classes: 10


In [14]:
print("Testing one batch...")

model.eval()

images, labels = next(iter(test_loader))

print("Batch shape:", images.shape)
print("Labels shape:", labels.shape)

start = time.time()

with torch.inference_mode():
    outputs = model(images)

elapsed = time.time() - start

print(f"Inference completed in {elapsed:.2f} seconds")
print("Output shape:", outputs.shape)

Testing one batch...
Batch shape: torch.Size([1, 3, 224, 224])
Labels shape: torch.Size([1])
Inference completed in 0.16 seconds
Output shape: torch.Size([1, 10])


In [15]:
# ============================================================
# SIMPLE CPU BENCHMARK TEST
# Tests only 20 images first
# ============================================================

import time
import torch
from tqdm import tqdm

print("=" * 60)
print("CPU BENCHMARK TEST")
print("=" * 60)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

PRECISION = "fp32"
MAX_IMAGES = 20

device = torch.device("cpu")

print(f"Precision: {PRECISION.upper()}")
print(f"Device:    {device}")
print()

# ------------------------------------------------------------
# Check that required objects exist
# ------------------------------------------------------------

print("Checking required objects...")

if "load_variant" not in globals():
    raise RuntimeError(
        "load_variant() is not defined. "
        "Run the cell containing load_variant() first."
    )

if "test_loader" not in globals():
    raise RuntimeError(
        "test_loader is not defined. "
        "Run the cell that creates test_loader first."
    )

print("✓ load_variant() found")
print("✓ test_loader found")
print()

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

print("Loading FP32 model...", flush=True)

load_start = time.perf_counter()

model, classes = load_variant(
    PRECISION,
    device
)

model.eval()

load_time = time.perf_counter() - load_start

print(f"✓ Model loaded in {load_time:.2f} seconds")
print(f"✓ Model type: {type(model).__name__}")
print()

# ------------------------------------------------------------
# Run benchmark
# ------------------------------------------------------------

print(f"Running first {MAX_IMAGES} images...")
print()

correct = 0
total = 0

start_time = time.perf_counter()

with torch.inference_mode():

    progress = tqdm(
        test_loader,
        desc="FP32 inference",
        unit="batch",
        dynamic_ncols=True
    )

    for images, labels in progress:

        images = images.to(device)
        labels = labels.to(device)

        # FP32
        images = images.float()

        # Inference
        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        # Timing
        elapsed = time.perf_counter() - start_time

        throughput = (
            total / elapsed
            if elapsed > 0
            else 0
        )

        accuracy = (
            correct / total
            if total > 0
            else 0
        )

        progress.set_postfix(
            images=f"{total}/{MAX_IMAGES}",
            accuracy=f"{accuracy:.3f}",
            speed=f"{throughput:.2f} img/s"
        )

        # Stop after requested number of images
        if total >= MAX_IMAGES:
            break

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

total_time = time.perf_counter() - start_time

accuracy = correct / total
throughput = total / total_time
latency_ms = (total_time / total) * 1000

print()
print("=" * 60)
print("BENCHMARK COMPLETE")
print("=" * 60)

print(f"Images:      {total}")
print(f"Correct:     {correct}")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Time:        {total_time:.2f} sec")
print(f"Latency:     {latency_ms:.2f} ms/image")
print(f"Throughput:  {throughput:.2f} images/sec")

print("=" * 60)

CPU BENCHMARK TEST
Precision: FP32
Device:    cpu

Checking required objects...
✓ load_variant() found
✓ test_loader found

Loading FP32 model...
✓ Model loaded in 0.31 seconds
✓ Model type: ResNet

Running first 20 images...



FP32 inference:   0%|             | 19/4050 [00:02<09:36,  6.99batch/s, accuracy=0.950, images=20/20, speed=7.34 img/s]


BENCHMARK COMPLETE
Images:      20
Correct:     19
Accuracy:    0.9500
Time:        2.73 sec
Latency:     136.34 ms/image
Throughput:  7.33 images/sec


In [17]:
print("TEST CELL STARTED")

import time
time.sleep(3)

print("TEST CELL FINISHED")

TEST CELL STARTED
TEST CELL FINISHED


In [18]:
print("BENCHMARK TEST STARTED")

import torch

print("PyTorch:", torch.__version__)
print("Device:", torch.device("cpu"))
print("CPU test:", torch.randn(1000, 1000).matmul(torch.randn(1000, 1000)).shape)

print("BENCHMARK TEST FINISHED")

BENCHMARK TEST STARTED
PyTorch: 2.11.0+cpu
Device: cpu
CPU test: torch.Size([1000, 1000])
BENCHMARK TEST FINISHED


In [20]:
print("Checking load_variant output...")

result = load_variant("fp32", torch.device("cpu"))

print("Type:", type(result))
print("Length:", len(result))

for i, item in enumerate(result):
    print(f"\nItem {i}:")
    print("Type:", type(item))
    print("Value:", item if not isinstance(item, torch.nn.Module) else "PyTorch Model")

Checking load_variant output...
Type: <class 'tuple'>
Length: 2

Item 0:
Type: <class 'torchvision.models.resnet.ResNet'>
Value: PyTorch Model

Item 1:
Type: <class 'list'>
Value: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


In [21]:
print("=" * 60)
print("FP32 ONE-IMAGE TEST")
print("=" * 60)

import time
import torch

device = torch.device("cpu")

print("Step 1: Loading model...")

model, class_names = load_variant("fp32", device)

print("Step 2: Model loaded")
print("Model type:", type(model))
print("Number of classes:", len(class_names))
print("Classes:", class_names)

model.eval()

print("Step 3: Getting one image...")

images, labels = next(iter(test_loader))

print("Image shape:", images.shape)
print("Label:", labels.item())

images = images.to(device).float()

print("Step 4: Running inference...")

start = time.perf_counter()

with torch.inference_mode():
    output = model(images)

elapsed = time.perf_counter() - start

prediction = output.argmax(dim=1).item()

print("Step 5: Inference complete")
print()
print("Prediction:", prediction)
print("Predicted class:", class_names[prediction])
print("Actual class:", class_names[labels.item()])
print(f"Inference time: {elapsed:.4f} seconds")

print("=" * 60)
print("TEST FINISHED SUCCESSFULLY")
print("=" * 60)

FP32 ONE-IMAGE TEST
Step 1: Loading model...
Step 2: Model loaded
Model type: <class 'torchvision.models.resnet.ResNet'>
Number of classes: 10
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Step 3: Getting one image...
Image shape: torch.Size([1, 3, 224, 224])
Label: 2
Step 4: Running inference...
Step 5: Inference complete

Prediction: 2
Predicted class: HerbaceousVegetation
Actual class: HerbaceousVegetation
Inference time: 0.1745 seconds
TEST FINISHED SUCCESSFULLY


In [23]:
import time
import torch
import pandas as pd
from tqdm import tqdm
from pathlib import Path

print("=" * 70)
print("FP32 CPU BENCHMARK WITH CHECKPOINTING")
print("=" * 70)

device = torch.device("cpu")

# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------
CHECKPOINT_FILE = Path("fp32_benchmark_progress.pt")
FINAL_CSV = Path("fp32_benchmark_result.csv")
SAVE_EVERY = 100

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------
print("\nLoading FP32 model...")

model, class_names = load_variant("fp32", device)
model.eval()

print("Model loaded successfully")
print("Classes:", len(class_names))
print("Test images:", len(test_loader.dataset))

# ---------------------------------------------------------
# Check for previous progress
# ---------------------------------------------------------
if CHECKPOINT_FILE.exists():

    print("\nPrevious benchmark progress found.")
    checkpoint = torch.load(
        CHECKPOINT_FILE,
        map_location="cpu",
        weights_only=False
    )

    start_index = checkpoint["images_processed"]
    correct = checkpoint["correct"]
    total_inference_time = checkpoint["inference_time"]

    print(f"Resuming from image: {start_index}")
    print(f"Correct so far:      {correct}")
    print(f"Accuracy so far:     {correct / start_index:.4f}")

else:

    start_index = 0
    correct = 0
    total_inference_time = 0.0

    print("\nStarting new benchmark.")

# ---------------------------------------------------------
# Warm-up only for fresh benchmark
# ---------------------------------------------------------
if start_index == 0:

    print("\nRunning warm-up...")

    images, labels = next(iter(test_loader))
    images = images.to(device).float()

    with torch.inference_mode():
        for _ in range(3):
            _ = model(images)

    print("Warm-up complete")

# ---------------------------------------------------------
# Benchmark
# ---------------------------------------------------------
print("\nStarting benchmark...\n")

start_total = time.perf_counter()

with torch.inference_mode():

    for batch_index, (images, labels) in enumerate(
        tqdm(
            test_loader,
            total=len(test_loader),
            desc="FP32 inference",
            unit="batch"
        )
    ):

        # Skip images already processed
        if batch_index < start_index:
            continue

        images = images.to(device).float()
        labels = labels.to(device)

        start = time.perf_counter()

        outputs = model(images)

        elapsed = time.perf_counter() - start

        predictions = outputs.argmax(dim=1)

        batch_size = images.size(0)

        correct += (predictions == labels).sum().item()
        start_index += batch_size
        total_inference_time += elapsed

        # -------------------------------------------------
        # Save checkpoint
        # -------------------------------------------------
        if start_index % SAVE_EVERY == 0:

            torch.save(
                {
                    "images_processed": start_index,
                    "correct": correct,
                    "inference_time": total_inference_time
                },
                CHECKPOINT_FILE
            )

            accuracy = correct / start_index
            throughput = start_index / total_inference_time

            print(
                f"\nCHECKPOINT SAVED | "
                f"Images: {start_index}/{len(test_loader.dataset)} | "
                f"Accuracy: {accuracy:.4f} | "
                f"Speed: {throughput:.2f} img/s"
            )

# ---------------------------------------------------------
# Final calculations
# ---------------------------------------------------------
total_wall_time = time.perf_counter() - start_total

accuracy = correct / start_index

latency_ms = (
    total_inference_time / start_index
) * 1000

throughput_ips = (
    start_index / total_inference_time
)

wall_throughput = (
    start_index / total_wall_time
)

# ---------------------------------------------------------
# Final result
# ---------------------------------------------------------
results = {
    "precision": "FP32",
    "images": start_index,
    "correct": correct,
    "accuracy": accuracy,
    "latency_ms": latency_ms,
    "inference_throughput_ips": throughput_ips,
    "wall_throughput_ips": wall_throughput,
    "inference_time_sec": total_inference_time,
    "wall_time_sec": total_wall_time
}

pd.DataFrame([results]).to_csv(
    FINAL_CSV,
    index=False
)

# ---------------------------------------------------------
# Remove checkpoint after successful completion
# ---------------------------------------------------------
if CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()

print("\n")
print("=" * 70)
print("FP32 BENCHMARK COMPLETE")
print("=" * 70)

print(f"Images evaluated:       {start_index}")
print(f"Correct predictions:    {correct}")
print(f"Accuracy:               {accuracy:.4f} ({accuracy * 100:.2f}%)")

print(f"\nPure inference time:    {total_inference_time:.2f} sec")
print(f"Total wall time:        {total_wall_time:.2f} sec")

print(f"\nInference latency:      {latency_ms:.2f} ms/image")
print(f"Inference throughput:   {throughput_ips:.2f} images/sec")
print(f"Wall-clock throughput:  {wall_throughput:.2f} images/sec")

print(f"\nResults saved to: {FINAL_CSV}")

print("=" * 70)

FP32 CPU BENCHMARK WITH CHECKPOINTING

Loading FP32 model...
Model loaded successfully
Classes: 10
Test images: 4050

Starting new benchmark.

Running warm-up...
Warm-up complete

Starting benchmark...



FP32 inference:   2%|█▍                                                          | 100/4050 [00:16<11:03,  5.95batch/s]


CHECKPOINT SAVED | Images: 100/4050 | Accuracy: 0.9700 | Speed: 6.27 img/s


FP32 inference:   5%|██▉                                                         | 201/4050 [00:33<09:54,  6.47batch/s]


CHECKPOINT SAVED | Images: 200/4050 | Accuracy: 0.9800 | Speed: 6.15 img/s


...


CHECKPOINT SAVED | Images: 3900/4050 | Accuracy: 0.9841 | Speed: 6.53 img/s


FP32 inference:  99%|██████████████████████████████████████████████████████████▎| 4001/4050 [10:34<00:07,  6.48batch/s]


CHECKPOINT SAVED | Images: 4000/4050 | Accuracy: 0.9845 | Speed: 6.54 img/s


FP32 inference: 100%|███████████████████████████████████████████████████████████| 4050/4050 [10:42<00:00,  6.31batch/s]




FP32 BENCHMARK COMPLETE
Images evaluated:       4050
Correct predictions:    3988
Accuracy:               0.9847 (98.47%)

Pure inference time:    619.45 sec
Total wall time:        642.14 sec

Inference latency:      152.95 ms/image
Inference throughput:   6.54 images/sec
Wall-clock throughput:  6.31 images/sec

Results saved to: fp32_benchmark_result.csv


## Cell 8 — Display results

In [25]:
import pandas as pd

results = pd.read_csv("fp32_benchmark_result.csv")

summary = pd.DataFrame({
    "Metric": [
        "Precision",
        "Images Evaluated",
        "Correct Predictions",
        "Accuracy",
        "Pure Inference Time",
        "Total Wall Time",
        "Inference Latency",
        "Inference Throughput",
        "Wall-Clock Throughput"
    ],
    "Result": [
        results.loc[0, "precision"],
        f"{int(results.loc[0, 'images']):,}",
        f"{int(results.loc[0, 'correct']):,}",
        f"{results.loc[0, 'accuracy'] * 100:.2f}%",
        f"{results.loc[0, 'inference_time_sec']:.2f} sec",
        f"{results.loc[0, 'wall_time_sec']:.2f} sec",
        f"{results.loc[0, 'latency_ms']:.2f} ms/image",
        f"{results.loc[0, 'inference_throughput_ips']:.2f} images/sec",
        f"{results.loc[0, 'wall_throughput_ips']:.2f} images/sec"
    ]
})

display(summary)

,Metric,Result
0,Precision,FP32
1,Images Evaluated,"4,050"
2,Correct Predictions,"3,988"
3,Accuracy,98.47%
4,Pure Inference Time,619.45 sec
5,Total Wall Time,642.14 sec
6,Inference Latency,152.95 ms/image
7,Inference Throughput,6.54 images/sec
8,Wall-Clock Throughput,6.31 images/sec
